In [98]:
import numpy as np
from collections import Counter

def recurrence_matrix(state_space):
    M = state_space.shape[0]
    dist_matrix = np.linalg.norm(
        state_space[:, None, :] - state_space[None, :, :], axis=2
    )
    print("Distance Matrix:\n", dist_matrix)
    epsilon = 0.1 * np.max(dist_matrix)
    print("Epsilon:", epsilon)
    R = _heavside_fn(epsilon - dist_matrix)
    return R

def _heavside_fn(x):
    return np.where(x <= 0, 0, 1)

# u(i) = {x(i), x(i+τ), x(i+2τ), ..., x(i+(m-1)τ)}
def state_space_reconstruction(data, m , tau):
    N = len(data)
    M = N - (m - 1) * tau
    state_space = np.zeros((M, m))
    for i in range(M):
        state_space[i] = data[i:i + m * tau:tau]
    return state_space

def recurrence_rate(R, exclude_loi = True):
    if exclude_loi:
        M = R.shape[0]
        return (np.sum(R)- np.trace(R)) / (R.size - M)
    return np.sum(R) / R.size

def _lengths_of_consecutive_ones(arr):
    array = np.asarray(arr, dtype=np.int8)
    if array.size == 0:
        return np.array([], dtype=int)
    edges = np.diff(np.r_[0, array, 0])
    starts = np.flatnonzero(edges == 1)
    ends = np.flatnonzero(edges == -1)
    return (ends - starts).tolist()

def p_of_l(R, exclude_loi = True):
    M = R.shape[0]
    Rb = (np.asarray(R) != 0).astype(np.uint8)
    if exclude_loi:
        Rb = Rb.copy()
        np.fill_diagonal(Rb, 0)
    counts = Counter()
    for k in range(-(M - 1), M):
        diag = np.diagonal(Rb, offset=k)
        print(f"Offset {k} gives {list(diag)}")
        for L in _lengths_of_consecutive_ones(diag):
            counts[L] += 1
    return dict(counts)

def determinism(R, l_min=2, exclude_loi=True):
    if exclude_loi:
        R = R.copy()
        np.fill_diagonal(R, 0)
    p = p_of_l(R, exclude_loi=False)
    R_sum = R.sum()
    det = (sum(L * count for L, count in p.items() if L >= l_min) / R_sum) if R_sum else np.nan
    return det, p, R_sum

def entropy(R, l_min=2, exclude_loi=True):
    if exclude_loi:
        R = R.copy()
        np.fill_diagonal(R, 0)
    counts = p_of_l(R, exclude_loi=False)
    counts = {l: c for l, c in counts.items() if l >= l_min}
    print("Counts for entropy calculation:", counts)
    total_counts = sum(counts.values())
    if total_counts == 0:
        return np.nan
    probs = {l: c / total_counts for l, c in counts.items()}
    print("Probabilities for entropy calculation:", probs)
    ENTR = -sum(p * np.log(p) for p in probs.values())
    return ENTR

def L_mean(R, l_min=2, exclude_loi=True):
    if exclude_loi:
        R = R.copy()
        np.fill_diagonal(R, 0)
    counts = p_of_l(R, exclude_loi=False)
    counts = {l: c for l, c in counts.items() if l >= l_min}
    print("Counts for L_mean calculation:", counts)
    total_counts = sum(counts.values())
    print("Total counts for L_mean calculation:", total_counts)
    if total_counts == 0:
        return np.nan
    L_mean = sum(L * c for L, c in counts.items()) / total_counts
    return L_mean

def L_max(R, l_min=2, exclude_loi=True):
    if exclude_loi:
        R = R.copy()
        np.fill_diagonal(R, 0)
    counts = p_of_l(R, exclude_loi=False)
    counts = {l: c for l, c in counts.items() if l >= l_min}
    print("Counts for L_max calculation:", counts)
    if not counts:
        return np.nan
    L_max = max(counts.keys())
    return L_max

def vertical_run_lengths_in_columns(col):
    runs = []
    run = 0
    for x in col:
        if x:
            run += 1
        else:
            if run > 0:
                runs.append(run)
                run =0
    if run > 0:
        runs.append(run)
    return runs

def p_of_v(R, func, exclude_loi=True):
    M = R.shape[0]
    Rb = (np.asarray(R) != 0).astype(np.uint8)
    if exclude_loi:
        Rb = Rb.copy()
        np.fill_diagonal(Rb, 0)
    counts = {}
    for j in range(M):  # each column
        col = Rb[:, j]
        runs = func(col)
        print(f"Column {j} runs: {runs}")
        for v in runs:
            counts[v] = counts.get(v, 0) + 1
    return counts

def trapping_time(R, v_min=2, exclude_loi=True):
    counts = p_of_v(R, func=_lengths_of_consecutive_ones, exclude_loi=exclude_loi)
    if not counts:
        return np.nan
    print("Counts of vertical runs for trapping time:", counts)
    counts = {v: c for v, c in counts.items() if v >= v_min}
    numer = sum(v * c for v, c in counts.items())
    print("Numerator for trapping time:", numer)
    denom = sum(counts.values())
    print("Denominator for trapping time:", denom)
    return numer / denom

def laminarity(R, v_min=2, exclude_loi=True):
    counts = p_of_v(R, func=_lengths_of_consecutive_ones, exclude_loi=exclude_loi)
    if not counts:
        return np.nan
    print(counts)
    numer = sum(v * c for v, c in counts.items() if v >= v_min)
    print("Numerator for laminarity:", numer)
    denom = sum(v * c for v, c in counts.items())
    print("Denominator for laminarity:", denom)
    if denom == 0:
        return np.nan
    return numer / denom

In [2]:
data = np.arange(1, 11)   # [1,2,3,4,5,6,7,8,9,10]

m = 3   # embedding dimension
tau = 2 # delay

reconstructed = state_space_reconstruction(data, m, tau)
    
print("Original data:", data)
print(f"\nReconstructed state space (m={m}, tau={tau}):\n")
print(reconstructed)

Original data: [ 1  2  3  4  5  6  7  8  9 10]

Reconstructed state space (m=3, tau=2):

[[ 1.  3.  5.]
 [ 2.  4.  6.]
 [ 3.  5.  7.]
 [ 4.  6.  8.]
 [ 5.  7.  9.]
 [ 6.  8. 10.]]


In [20]:
R = recurrence_matrix(reconstructed)
print(f"\nRecurrence Matrix (R):\n{R}")


Distance Matrix:
 [[0.         1.73205081 3.46410162 5.19615242 6.92820323 8.66025404]
 [1.73205081 0.         1.73205081 3.46410162 5.19615242 6.92820323]
 [3.46410162 1.73205081 0.         1.73205081 3.46410162 5.19615242]
 [5.19615242 3.46410162 1.73205081 0.         1.73205081 3.46410162]
 [6.92820323 5.19615242 3.46410162 1.73205081 0.         1.73205081]
 [8.66025404 6.92820323 5.19615242 3.46410162 1.73205081 0.        ]]
Epsilon: 0.8660254037844388

Recurrence Matrix (R):
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]]


In [8]:
recurrence_rate_with_diag = recurrence_rate(R, exclude_loi=False)
print(f"\nRecurrence Rate (including LOI): {recurrence_rate_with_diag}")
recurrence_rate_without_diag = recurrence_rate(R, exclude_loi=True)
print(f"\nRecurrence Rate (excluding LOI): {recurrence_rate_without_diag}")


Recurrence Rate (including LOI): 0.16666666666666666

Recurrence Rate (excluding LOI): 0.0


In [9]:
from collections import Counter

# Counting elements in a list
data = ['apple', 'banana', 'apple', 'orange', 'banana', 'apple']
fruit_counts = Counter(data)
print(fruit_counts)

# Accessing a count
print(f"Count of 'apple': {fruit_counts['apple']}")
print(f"Count of 'grape': {fruit_counts['grape']}") # Returns 0 for missing items

# Finding the most common elements
print(f"Most common fruits: {fruit_counts.most_common(2)}")

Counter({'apple': 3, 'banana': 2, 'orange': 1})
Count of 'apple': 3
Count of 'grape': 0
Most common fruits: [('apple', 3), ('banana', 2)]


In [18]:
print(R)
p_of_l(R, exclude_loi=True)


[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]]
Offset -5 gives [np.uint8(0)]
Offset -4 gives [np.uint8(0), np.uint8(0)]
Offset -3 gives [np.uint8(0), np.uint8(0), np.uint8(0)]
Offset -2 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset -1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 0 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 2 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 3 gives [np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 4 gives [np.uint8(0), np.uint8(0)]
Offset 5 gives [np.uint8(0)]


In [56]:
array = [0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1]
diff = np.diff(np.r_[0, array, 0])
print("Diff array:", diff)
start = np.flatnonzero(diff == 1)
print("Start indices:", start)
end = np.flatnonzero(diff == -1)
print("End indices:", end)
consective_ones  = (end - start).tolist()
print("Lengths of consecutive ones:", consective_ones)
counts = Counter()
for l in consective_ones:
    counts[l] += 1
print("Counts of lengths:", dict(counts))
print("Items in counts:", dict(counts).items())
print("Sum of counts:", sum(counts.values()))


Diff array: [ 0  1 -1  0  1  0 -1  1  0  0 -1  1  0 -1]
Start indices: [ 1  4  7 11]
End indices: [ 2  6 10 13]
Lengths of consecutive ones: [1, 2, 3, 2]
Counts of lengths: {1: 1, 2: 2, 3: 1}
Items in counts: dict_items([(1, 1), (2, 2), (3, 1)])
Sum of counts: 4


In [32]:
R_sum = R.sum()
print("R_sum:", R_sum)

R_sum: 6


In [51]:
det, p, R_sum = determinism(R)
print("Determinism:", det)
print("Items in p:", p)
print("Sum of R:", R_sum)

Offset -5 gives [np.uint8(0)]
Offset -4 gives [np.uint8(0), np.uint8(0)]
Offset -3 gives [np.uint8(0), np.uint8(0), np.uint8(0)]
Offset -2 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset -1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 0 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 2 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 3 gives [np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 4 gives [np.uint8(0), np.uint8(0)]
Offset 5 gives [np.uint8(0)]
Determinism: nan
Items in p: {}
Sum of R: 0


In [71]:
another_R = np.array([
        [1, 0, 1, 1, 0],
        [0, 1, 0, 1, 1],
        [1, 0, 1, 0, 1],
        [1, 1, 0, 1, 0],
        [1, 0, 1, 0, 1]
    ], dtype=np.uint8)

det2, p2, R_sum = determinism(another_R)
print("Determinism (another_R):", det2)
print("Items in p2:", p2)
print("Sum of R (another_R):", R_sum)

Offset -4 gives [np.uint8(1)]
Offset -3 gives [np.uint8(1), np.uint8(0)]
Offset -2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset -1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 0 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset 3 gives [np.uint8(1), np.uint8(1)]
Offset 4 gives [np.uint8(0)]
Determinism (another_R): 0.8
Items in p2: {1: 2, 3: 2, 2: 1}
Sum of R (another_R): 10


In [ ]:
another_R_sum = another_R.sum()
print("another_R_sum:", another_R_sum)

another_R_sum: 14


In [60]:
entr = entropy(another_R, l_min=2, exclude_loi=True)
print("Entropy:", entr)

Offset -4 gives [np.uint8(1)]
Offset -3 gives [np.uint8(0), np.uint8(0)]
Offset -2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset -1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 0 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset 3 gives [np.uint8(1), np.uint8(1)]
Offset 4 gives [np.uint8(0)]
Counts for entropy calculation: {3: 2, 2: 1}
Probabilities for entropy calculation: {3: 0.6666666666666666, 2: 0.3333333333333333}
Entropy: 0.6365141682948128


In [65]:
l_mean = L_mean(another_R, l_min=2, exclude_loi=True)
print("L_mean:", l_mean)

Offset -4 gives [np.uint8(1)]
Offset -3 gives [np.uint8(0), np.uint8(0)]
Offset -2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset -1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 0 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 1 gives [np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(0)]
Offset 2 gives [np.uint8(1), np.uint8(1), np.uint8(1)]
Offset 3 gives [np.uint8(1), np.uint8(1)]
Offset 4 gives [np.uint8(0)]
Counts for L_mean calculation: {3: 2, 2: 1}
Total counts for L_mean calculation: 3
L_mean: 2.6666666666666665


In [72]:
print(another_R)
counts = p_of_v(another_R, _lengths_of_consecutive_ones, v_min=2, exclude_loi=True)
print("Counts of vertical runs:", counts)

[[1 0 1 1 0]
 [0 1 0 1 1]
 [1 0 1 0 1]
 [1 1 0 1 0]
 [1 0 1 0 1]]
Column 0 runs: [3]
Column 1 runs: [1]
Column 2 runs: [1, 1]
Column 3 runs: [2]
Column 4 runs: [2]
Counts of vertical runs: {3: 1, 2: 2}


In [95]:
trapping_time = trapping_time(another_R, v_min=2, exclude_loi=True)
print("Trapping Time:", trapping_time)

Column 0 runs: [3]
Column 1 runs: [1]
Column 2 runs: [1, 1]
Column 3 runs: [2]
Column 4 runs: [2]
Counts of vertical runs for trapping time: {3: 1, 1: 3, 2: 2}
Numerator for trapping time: 7
Denominator for trapping time: 3
Trapping Time: 2.3333333333333335


In [99]:
laminarity = laminarity(another_R, v_min=2, exclude_loi=True)
print("Laminarity:", laminarity)

Column 0 runs: [3]
Column 1 runs: [1]
Column 2 runs: [1, 1]
Column 3 runs: [2]
Column 4 runs: [2]
{3: 1, 1: 3, 2: 2}
Numerator for laminarity: 7
Denominator for laminarity: 10
Laminarity: 0.7
